# Combine Dataset: ISOT Fake and Real News

This notebook combines the two raw files into a single dataset for the rest of the project.

- `Dataset/True.csv`: real articles, collected from Reuters
- `Dataset/Fake.csv`: fake articles, collected from sites flagged as unreliable

Each file gets a `label` column, and the two are stacked into one table. No cleaning happens here. Data quality is checked in `dataset_eda.ipynb` and fixed in `clean_dataset.ipynb`, so the combined file stays a faithful copy of the raw data.

## 1. Setup

In [1]:
import pandas as pd

from preprocessing import RANDOM_STATE

## 2. Load the Raw Files

In [2]:
true_df = pd.read_csv("Dataset/True.csv")
fake_df = pd.read_csv("Dataset/Fake.csv")

print("True.csv:", true_df.shape)
print("Fake.csv:", fake_df.shape)
print("\nColumns match:", list(true_df.columns) == list(fake_df.columns))
print("Columns:", list(true_df.columns))

True.csv: (21417, 4)
Fake.csv: (23481, 4)

Columns match: True
Columns: ['title', 'text', 'subject', 'date']


## 3. Add Labels

The label comes from which file an article is in. `label` keeps the readable name, and `label_encoded` is the numeric target used by every model: `fake = 1`, `real = 0`, so Fake-class recall is measured on class 1.

In [3]:
true_df["label"] = "real"
fake_df["label"] = "fake"

## 4. Combine and Shuffle

The two tables are stacked, then shuffled, so that the rows are not all real followed by all fake. The shuffle does not affect the models, since `get_split_tfidf()` already shuffles and stratifies, but it makes `df.head()` show a mix of both classes.

In [4]:
df = pd.concat([true_df, fake_df], ignore_index=True)
df["label_encoded"] = df["label"].map({"fake": 1, "real": 0})

df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
df.head()

,title,text,subject,date,label,label_encoded
0,BREAKING: GOP Chairman Grassley Has Had Enoug...,"Donald Trump s White House is in chaos, and th...",News,"July 21, 2017",fake,1
1,Failed GOP Candidates Remembered In Hilarious...,Now that Donald Trump is the presumptive GOP n...,News,"May 7, 2016",fake,1
2,Mike Pence’s New DC Neighbors Are HILARIOUSLY...,Mike Pence is a huge homophobe. He supports ex...,News,"December 3, 2016",fake,1
3,California AG pledges to defend birth control ...,SAN FRANCISCO (Reuters) - California Attorney ...,politicsNews,"October 6, 2017",real,0
4,AZ RANCHERS Living On US-Mexico Border Destroy...,Twisted reasoning is all that comes from Pelos...,politics,"Apr 25, 2017",fake,1


## 5. Sanity Checks

Confirm that no rows were lost, that every row has a label, and that the mapping is correct.

In [5]:
print("Combined shape:", df.shape)
print("Rows preserved:", len(df) == len(true_df) + len(fake_df))
print("Missing labels:", df["label_encoded"].isna().sum())
print()
print(df["label"].value_counts())
print()
print(df[["label", "label_encoded"]].drop_duplicates())

Combined shape: (44898, 6)
Rows preserved: True
Missing labels: 0

label
fake    23481
real    21417
Name: count, dtype: int64

  label  label_encoded
0  fake              1
3  real              0


## 6. Save

The combined file is saved gzip-compressed. As plain CSV it is about 117 MB, which is over GitHub's 100 MB file limit. Compressed it is about 45 MB, and `pd.read_csv("Dataset/fake_news_combined.csv.gz")` reads it directly, with no extra step.

In [6]:
df.to_csv("Dataset/fake_news_combined.csv.gz", index=False, compression="gzip")
print("Saved combined dataset to Dataset/fake_news_combined.csv.gz")

Saved combined dataset to Dataset/fake_news_combined.csv.gz
